# Batch-generate rule cube visualizations

This notebook saves interactive cube visualizations for approximate 4-hour event windows within the first 10,000 timestamp-ordered events among the most frequent event types. Figures are written to `Results/Visualizations` and are not displayed inline.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "Source"))

from spatio_temporal_network import (
    LearningParameters,
    OnlineSTDRMiner,
    initialize_network_from_events,
)
from visualize_rule_cube import prepare_osm_basemap_trace, visualize_significant_synapse_cube

def safe_time_label(value):
    timestamp = pd.to_datetime(value, errors="coerce")
    if pd.notna(timestamp):
        return timestamp.strftime("%Y%m%d_%H%M%S")
    return "".join(char if char.isalnum() else "_" for char in str(value)).strip("_")


def batch_timespan_filename(batch_events):
    if "Occurrence_time" in batch_events.columns:
        start_label = safe_time_label(batch_events["Occurrence_time"].iloc[0])
        end_label = safe_time_label(batch_events["Occurrence_time"].iloc[-1])
    else:
        start_label = f"t{float(batch_events['timestamp'].iloc[0]):.2f}".replace(".", "p")
        end_label = f"t{float(batch_events['timestamp'].iloc[-1]):.2f}".replace(".", "p")
    return f"network_cube_{start_label}_to_{end_label}.html"


Load the dataset, keep the 50 most frequent event types, then take the first 10,000 events from that filtered stream.

In [ ]:
csv_path = PROJECT_ROOT / "Data" / "Preprocessed" / "dataset_TSMC2014_NYC_preprocessed.csv"
events = pd.read_csv(csv_path).sort_values("timestamp")

top_event_type_count = 10
max_events = 5000
selected_event_types = events["Event_type"].value_counts().head(top_event_type_count).index.tolist()
selected_events = events[events["Event_type"].isin(selected_event_types)].head(max_events).copy()

print("selected event types:", len(selected_event_types))
print("selected events:", len(selected_events))
selected_events["Event_type"].value_counts().head(20)


Configure network, learning, and visualization parameters. Checkpoints are selected by approximate 4-hour time windows using the `timestamp` column, which is measured in minutes.

In [ ]:
from math import log
window_hours = 3
window_minutes = window_hours * 60

event_numbers = []
window_start_indices = []
window_start_index = 0
window_start_time = float(selected_events["timestamp"].iloc[0])

for row_position, timestamp in enumerate(selected_events["timestamp"].to_numpy(), start=1):
    if float(timestamp) - window_start_time >= window_minutes:
        event_numbers.append(row_position)
        window_start_indices.append(window_start_index)
        window_start_index = row_position
        window_start_time = float(timestamp)

if not event_numbers or event_numbers[-1] != len(selected_events):
    event_numbers.append(len(selected_events))
    window_start_indices.append(window_start_index)

checkpoint_start_by_event = dict(zip(event_numbers, window_start_indices))

grid_shape = (10, 10)
spatial_threshold = 10_000.0
max_synapses_to_draw = 1000
show_basemap = True
show_all_event_points = False
basemap_zoom = 12
basemap_resolution = (128, 128)

parameters = LearningParameters(
    eta=1,
    lambda_decay=-log(0.10) / window_minutes,
    tau_activation=120,
    tau_refractory=12,
    theta=0.2,
)

basemap_trace_data = None
if show_basemap:
    try:
        basemap_trace_data = prepare_osm_basemap_trace(
            events=selected_events,
            zoom=basemap_zoom,
            resolution=basemap_resolution,
        )
    except Exception as exc:
        print(f"Basemap unavailable; continuing without it: {exc}")
        show_basemap = False

output_dir = PROJECT_ROOT / "Results" / "Visualizations"
output_dir.mkdir(parents=True, exist_ok=True)

for old_html in output_dir.glob("network_cube_*.html"):
    old_html.unlink()

checkpoint_preview = pd.DataFrame(
    {
        "event_number": event_numbers,
        "batch_start_event": [start + 1 for start in window_start_indices],
        "batch_end_event": event_numbers,
        "start_timestamp": [selected_events["timestamp"].iloc[start] for start in window_start_indices],
        "end_timestamp": [selected_events["timestamp"].iloc[end - 1] for end in event_numbers],
    }
)
checkpoint_preview["duration_hours"] = (
    checkpoint_preview["end_timestamp"] - checkpoint_preview["start_timestamp"]
) / 60.0
checkpoint_preview


In [ ]:
network = initialize_network_from_events(
    events=selected_events,
    grid_shape=grid_shape,
    spatial_threshold=spatial_threshold,
)
miner = OnlineSTDRMiner(network, parameters)

written_files = []
last_summary = None
checkpoint_set = set(event_numbers)

for event_index, event in enumerate(selected_events.itertuples(index=False), start=1):
    miner.process_event(pd.Series(event._asdict()), extract_rules=False)
    if event_index not in checkpoint_set:
        continue

    batch_start_index = checkpoint_start_by_event[event_index]
    batch_events = selected_events.iloc[batch_start_index:event_index].copy()
    output_html = output_dir / batch_timespan_filename(batch_events)
    fig = visualize_significant_synapse_cube(
        events=selected_events,
        network=network,
        miner=miner,
        processed_events=batch_events,
        output_html=output_html,
        show_all_event_points=show_all_event_points,
        show_basemap=show_basemap,
        basemap_zoom=basemap_zoom,
        basemap_resolution=basemap_resolution,
        basemap_trace_data=basemap_trace_data,
        max_synapses=max_synapses_to_draw,
    )
    written_files.append(output_html)
    last_summary = {
        "event_number": event_index,
        "batch_start_event": batch_start_index + 1,
        "batch_end_event": event_index,
        "significant_synapses": len(miner.significant_synapses),
        "network": network.summary(),
    }
    duration_hours = (batch_events["timestamp"].iloc[-1] - batch_events["timestamp"].iloc[0]) / 60.0
    print(f"wrote {output_html.name}; batch={batch_start_index + 1}-{event_index}; duration={duration_hours:.2f}h; significant synapses={len(miner.significant_synapses)}")
    del fig

print("files written:", len(written_files))
print("output directory:", output_dir)
last_summary
